# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [2]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [3]:
df['revenue'] = df['qty'] * df['price']
print('Total Revenue: ', df['revenue'].sum())
print('Total Units:', df['qty'].sum())

Total Revenue:  8520.0
Total Units: 783


Interpretation: In 400 orders 783 units were sold and $8520.0 in revenue was generated.

Explanation: I started by adding revenue to the table by multiplying 'qty' and 'price'. I then printed the sum of 'revenue' and 'qty' and verified the total revenue and total units are what they should be.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [14]:
by_category = df.groupby('category', as_index = False)['revenue'].sum() 
by_category = by_category.sort_values('revenue', ascending = False)
by_category['percent'] = (100 * by_category['revenue'] / df['revenue'].sum()).round(1)
by_category

,category,revenue,percent
1,Food,4293.0,50.4
2,Merch,1771.5,20.8
0,Drink,1554.0,18.2
3,RainGear,901.5,10.6


Interpretation: Food accounted for the highest percent (50.4%) of total revenue, generating $4293.0. Merch, Drink, and RainGear followed behind in that order for percent of revenue generated.

Explanation: I started by grouping all the orders by category and finding the total revenue for each category. I then sorted in descending order and found what percent each category represented of total revenue.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [5]:
vendorTable = df.groupby('vendor_id', as_index = False).agg(averageRevenue = ('revenue', 'mean'), orderCount = ('vendor_id', 'count')).round(2)
vendorTable = vendorTable.sort_values('averageRevenue', ascending = False)
vendorTable

,vendor_id,averageRevenue,orderCount
0,V-01,22.60,94
3,V-18,21.75,108
1,V-05,20.58,93
2,V-10,20.31,105


Interpretation: The vendor with ID V-01 had the highest average order revenue, with 94 orders at an average of $22.60 per order.

Explanation: I started by grouping the orders by vendor and then finding the average revenue and order count per vendor. I then sorted each vendor in descending order to find the highest average order revenue.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [6]:
merchPercent = 100 * df.loc[df['category'] == 'Merch', 'revenue'].sum() / df['revenue'].sum()
print(round(merchPercent, 1))

20.8


Interpretation: 20.8% of total revenue comes from Merch.

Explanation: I first found all Merch orders, summed up all their revenues, and divided that revenue by total revenue. I then multiplied by 100 to get the percent Merch makes up of total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [7]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on = 'vendor_id', how = 'left', validate = 'many_to_one')
print('Row count before: ', len(df), ' Row count after: ', len(joined)) 
print('Revenue total before: ', df['revenue'].sum(), ' Revenue total after: ', joined['revenue'].sum())
print(joined[joined['vendor_name'].isna()]['vendor_id'].unique())
joined['vendor_name'] = joined['vendor_name'].fillna('Missing vendor')

Row count before:  400  Row count after:  400
Revenue total before:  8520.0  Revenue total after:  8520.0
['V-18']


Interpretation: The row count (400) and revenue total (8520.0) stayed the same after the merge.

**The unmatched vendor, and what I did about it:**
The unmatched vendor was V-18. I decided to keep it with the other orders and set its 'vendor_name' to 'Missing vendor' so it is properly displayed. 

Explanation: I used left join to bring in vendor names and then verified that the row count and revenue total stayed the same by printing each value. I then found the vendor missing a name and named it 'Missing vendor'.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [8]:
pivotTable = pd.pivot_table(joined, index = 'vendor_name', columns = 'category', values = 'revenue', aggfunc = 'sum', margins = True, margins_name = 'Total')
pivotTable

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Missing vendor,582.0,1018.5,508.5,240.0,2349.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


Interpretation: The total revenue per category per vendor is displayed in each cell with the grand totals in the last row and column across all vendors and categories. Food was the category with the highest total revenue and Missing vendor was the vendor with the highest total revenue. Grand total revenue was $8520.0 across all vendors and categories.

Explanation: I created a table by putting vendors as rows and categories as columns, then reported each vendor's revenue per category. I then created an extra row and column to total all previous values in each row/column.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [9]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell these vendors that they should make sure they have enough food and resources to provide food next game. I would recommend this because food was the highest earning category by a significant amount, producing $4,293 in revenue across all vendors. This accounts for 50.4% of total revenue, making food an extremely important consideration for these vendors. I would also recommend that vendors not bring as much rain gear next game (assuming weather is similar), as rain gear only produced $901.5 in revenue across all vendors. This makes up only 10.6% of total revenue, making it a category that wasn't too impactful on these vendors' profits. Overall, vendors should ensure they have adequate food supplies, bring less rain gear, and keep drinks and merch about the same for next game.

b) Of my seven answers I think question 6 is the least trustworthy. I think this because the highest earning vendor in this question was the one I had to label as 'Missing vendor'. Having the highest earning vendor also be the only unmatched vendor is not ideal for accurate and insightful data collection, making this a weakness. 